# 2 · Modelagem supervisionada

Pipeline completa de Machine Learning: tratamento, engenharia de atributos,
transformação, tratamento de vazamento, integração do pré-processamento ao modelo,
treinamento e validação.

**Dois desenhos**, porque são duas perguntas de generalização diferentes:

| | Modelo A — temporal | Modelo B — espacial |
|---|---|---|
| treino | 2023 | 75% dos municípios de 2024 |
| teste | 2024 | 25% dos municípios restantes |
| histórico t-1 | indisponível (não há 2022) | disponível |
| pergunta | sobrevive à passagem do tempo? | vale para municípios nunca vistos? |

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

# O acesso ao BigQuery exige o token de curta duração:
#   export GCP_ACCESS_TOKEN=$(gcloud auth print-access-token)

from src.modeling.dataset import carregar_abt, split_temporal, split_espacial
from src.preprocessing.features import ALVO, descartar_degeneradas, selecionar_features
from src.preprocessing.pipeline import resumo_preprocessamento
from src.modeling import candidatos, train

df = carregar_abt()
treino, teste = split_temporal(df)
len(treino), len(teste)

## Tratamento de data leakage

Três exclusões, todas medidas e não presumidas:

1. `proficiencia` — o alvo é **exatamente** `proficiencia >= 743`: 0 discordâncias em
   3,35 milhões de alunos.
2. `presenca` e `preenchimento_caderno` — determinam a classe negativa (513.401
   registros, 0% alfabetizados).
3. Todo contexto de desempenho é **defasado em um ano**. E as metas do Compromisso
   Nacional entram nessa regra: elas correlacionam **0,968** com o resultado de 2023,
   porque foram calculadas a partir dele.

In [ ]:
features = selecionar_features(list(df.columns), permitir_defasadas=False)
features, descartadas = descartar_degeneradas(treino, features)
print(f"{len(features)} features")
descartadas

## Pré-processamento integrado ao modelo

Imputação, escalonamento e encoding são **estimadores**: têm parâmetros aprendidos.
Aplicá-los ao dataframe inteiro antes do split faria a mediana de imputação e as
médias do target encoding enxergarem o teste. Dentro de um `Pipeline`, cada `fit`
acontece só na dobra de treino.

In [ ]:
resumo_preprocessamento(treino[features]).head(25)

## Os candidatos

Três famílias, para que a escolha final seja defendida por evidência: uma referência
trivial (piso), um modelo linear interpretável (contraprova) e ensembles de árvores
por boosting (estado da arte em dados tabulares).

In [ ]:
for nome, (est, escalonar, desc) in candidatos.CANDIDATOS.items():
    print(f"{nome:12} {type(est).__name__:32} {desc}")

## Modelo A — validação out-of-time

In [ ]:
resultado_a = train.executar("temporal")
resultado_a["metricas"].round(4)

### Por estrato de cobertura territorial

Três UFs (AC, DF, SP) aparecem só em 2024 — 23,1% do conjunto de teste. Um número
único misturaria *generalização temporal* com *extrapolação geográfica*.

In [ ]:
resultado_a["por_estrato"].round(4)

## Modelo B — validação espacial (municípios nunca vistos)

In [ ]:
resultado_b = train.executar("espacial")
resultado_b["metricas"].round(4)

## Comparação estatística

Duas AUCs diferentes não bastam: é preciso saber se a diferença excede a incerteza
amostral. O bootstrap **pareado** usa as mesmas reamostragens nos dois modelos,
eliminando a variância comum.

In [ ]:
from src.evaluation.metrics import comparar_bootstrap, ic_bootstrap

y = resultado_a["teste"][ALVO].astype(int).to_numpy()
probs = resultado_a["probabilidades"]
reais = [m for m in resultado_a["metricas"].index if m != "referencia"]
vencedor = reais[0]

for m in reais:
    v, lo, hi = ic_bootstrap(y, probs[m])
    print(f"{m:12} ROC AUC = {v:.4f}  IC 95% [{lo:.4f}, {hi:.4f}]")

print()
for m in reais[1:]:
    r = comparar_bootstrap(y, probs[m], probs[vencedor])
    print(f"{vencedor} - {m:12} = {r['diferenca']:+.4f}  p = {r['p_valor']:.3f}")

O resultado é um **empate estatístico**: a vantagem do LightGBM sobre a
regressão logística não é distinguível da incerteza amostral (p ≈ 0,09). O teto é
imposto pelos dados, não pelo algoritmo — ver §5 e §8.1 do README.

## Otimização de hiperparâmetros

A busca roda sob **validação cruzada agrupada por município**. Sem o agrupamento, o
hiperparâmetro vencedor seria o que melhor MEMORIZA a média municipal — o oposto do
objetivo. O espaço de busca é de regularização, não de capacidade.

In [ ]:
# Demora alguns minutos; descomente para executar.
# from src.modeling import tune
# melhor_config = tune.executar("temporal", n_tentativas=40)
# melhor_config